In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp


In [2]:
file_list = os.listdir(f"/media/ubuntu/sda/mouse_test/raw_data/WLF_128chmouse1_ASDstim_251202_181240")
file_list.remove("settings.xml")
#file_list.remove("log_200846.csv")
file_list = sorted(file_list)
recording_raw_list = []
for file in file_list:
    recording_raw_list.append(se.read_intan(f"/media/ubuntu/sda/mouse_test/raw_data/WLF_128chmouse1_ASDstim_251202_181240/{file}", stream_id= '0'))
recording_raw = concatenate_recordings(recording_list=recording_raw_list)

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_raw = spre.resample(recording_raw, 10000)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

In [3]:
probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
recording_f = recording_f.set_probegroup(probe)

In [4]:
output_folder = f'/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_ASDstim_251202_181240_10k_kilosort'
recording_preprocessed = recording_f.save(format="binary")

sorting_kilosort4 = ss.run_sorter(
    sorter_name="kilosort4", 
    recording=recording_preprocessed, 
    folder=output_folder + "/kilosort4"
)

analyzer_kilosort4 = si.create_sorting_analyzer(
    sorting=sorting_kilosort4, 
    recording=recording_preprocessed, 
    format='binary_folder', 
    folder=output_folder + '/analyzer_kilosort4_binary'
)

extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

qm_params = sqm.get_default_qm_params()
analyzer_kilosort4.compute("quality_metrics", qm_params)

sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

Use cache_folder=/tmp/spikeinterface_cache/tmpa58ksjtf/Z1EZ972F
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=10,000 - chunk_memory=2.44 MiB - total_memory=2.44 MiB - chunk_duration=1.00s


write_binary_recording (no parallelization):   0%|          | 0/8179 [00:00<?, ?it/s]

100%|██████████| 8/8 [04:13<00:00, 31.73s/it]


estimate_sparsity (no parallelization):   0%|          | 0/8179 [00:00<?, ?it/s]

compute_waveforms (no parallelization):   0%|          | 0/8179 [00:00<?, ?it/s]

...

spike_amplitudes (no parallelization):   0%|          | 0/8179 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/222 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/222 [00:00<?, ?it/s]

extract PCs (no parallelization):   0%|          | 0/8179 [00:00<?, ?it/s]

Run:
phy template-gui  /media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_ASDstim_251202_181240_10k_kilosort/phy_folder_for_kilosort/params.py


In [9]:
recording_f

CommonReferenceRecording: 128 channels - 10000.0Hz - 1 segments - 81,786,432 samples 
                          8,178.64s (2.27 hours) - int16 dtype - 19.50 GiB

In [6]:
sorting_kilosort4 = ss.run_sorter(
    sorter_name="kilosort4", 
    recording=recording_preprocessed, 
    folder=output_folder + "/kilosort4"
)

analyzer_kilosort4 = si.create_sorting_analyzer(
    sorting=sorting_kilosort4, 
    recording=recording_preprocessed, 
    format='binary_folder', 
    folder=output_folder + '/analyzer_kilosort4_binary'
)

extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

qm_params = sqm.get_default_qm_params()
analyzer_kilosort4.compute("quality_metrics", qm_params)

sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

ValueError: Folder /media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_ASDstim_251202_181240_10k_kilosort/kilosort4 already exists

In [ ]:
recording_f

CommonReferenceRecording: 128 channels - 10000.0Hz - 1 segments - 29,329,792 samples 
                          2,932.98s (48.88 minutes) - int16 dtype - 6.99 GiB

In [ ]:
extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

compute_waveforms (no parallelization):   0%|          | 0/1363 [00:00<?, ?it/s]

noise_level (no parallelization):   0%|          | 0/20 [00:00<?, ?it/s]

Compute : spike_locations (no parallelization):   0%|          | 0/1363 [00:00<?, ?it/s]

In [ ]:
sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

write_binary_recording (no parallelization):   0%|          | 0/1363 [00:00<?, ?it/s]

spike_amplitudes (no parallelization):   0%|          | 0/1363 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/149 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/149 [00:00<?, ?it/s]

extract PCs (no parallelization):   0%|          | 0/1363 [00:00<?, ?it/s]

Run:
phy template-gui  /media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251201_204035/phy_folder_for_kilosort/params.py
